In [1]:
import os
from pathlib import Path

import duckdb
import geopandas as gpd
import numpy as np
import pandas as pd

In [34]:
data_path = Path(os.environ["DATA_PATH"])
generated_path = data_path / "generated"
scripts_path = Path("./scripts")

In [3]:
with (scripts_path / "01_registro.sql").open(encoding="utf8") as f:
    registro_query = f.read()
    duckdb.sql(registro_query)

df_orig = (
    pd.read_parquet("data/processed/registro.parquet")
    # .dropna(subset=["partida_main"])
    # .assign(geometry=lambda df: gpd.points_from_xy(df["longitud"], df["latitud"]))
    # .drop(columns=["longitud", "latitud"])
    # .pipe(gpd.GeoDataFrame, geometry="geometry", crs="EPSG:4326")
    # .to_crs("EPSG:6372")
)

In [4]:
df_orig

,partida,folio,inmobiliaria,fecha_operacion,fecha_partida,valor_operacion,edad,lote,manzana,privada,direccion,acreedor,comprador,mts_superficie,latitud,longitud,fraccionamiento
0,5921278,1538428,Brasa,2021-02-11,2021-05-21,600000.0000,27.0,NaN,NaN,NaN,NaN,CONTADO,INDIVIDUO,NaN,NaN,NaN,LA RIOJA SECCION CASTILLAUNA
1,5898811,1538428,Brasa,2020-04-20,2020-07-24,893688.0625,NaN,NaN,NaN,NaN,NaN,CONTADO,PARCELAS CHUVISCAR,NaN,NaN,NaN,LA RIOJA SECCION CASTILLAUNA
2,6051425,334547,Casas Cadena,2024-12-30,2025-05-20,680000.0000,22.0,39,2,NaN,FRACCIONAMIENTO VALLE DEL PROGRESO,INFONAVIT,INDIVIDUO,120.000000,32.588318,-115.579407,Valle Del Progreso
3,6049768,334582,Casas Cadena,2024-12-23,2025-05-01,680000.0000,23.0,4,2,NaN,FRACCIONAMIENTO VALLE DEL PROGRESO,INFONAVIT,INDIVIDUO,120.000000,32.588318,-115.579407,Valle Del Progreso
4,6049447,334612,Casas Cadena,2024-12-23,2025-04-28,680000.0000,25.0,8,1,NaN,FRACCIONAMIENTO VALLE DEL PROGRESO,INFONAVIT,INDIVIDUO,120.000000,32.588318,-115.579407,Valle Del Progreso
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7132,5899756,387993,Ruba,2020-01-10,2020-08-01,426000.0000,NaN,23,41,NaN,FRACCIONAMIENTO PARAJES DE PUEBLA,INFONAVIT,INDIVIDUO,120.050003,32.557625,-115.346184,Parajes De Puebla
7133,5896400,388036,Ruba,2020-01-10,2020-05-29,426000.0000,NaN,44,40,NaN,FRACCIONAMIENTO PARAJES DE PUEBLA,INFONAVIT,INDIVIDUO,120.050003,32.557625,-115.346184,Parajes De Puebla
7134,5895108,1579352,Ruba,2020-01-10,2020-04-13,920000.0000,NaN,21,50,NaN,DESARROLLO URBANO PRIVADAS CAMPESTRE SEGUNDA S...,INFONAVIT,INDIVIDUO,122.500000,32.582203,-115.421379,Privadas Campestre
7135,5902030,360408,Ruba,2020-01-08,2020-09-18,483000.0000,NaN,8,62,NaN,FRACCIONAMIENTO ANGELES DE PUEBLA,INFONAVIT,RUVA,120.059998,32.564770,-115.339935,Angeles De Puebla


In [36]:
df_init = (
    pd.read_excel(
        data_path / "processing" / "RPPC_vivienda-interes-social-nueva_2020-2024.xlsx",
    )
    .rename(
        columns={
            "Inmobiliaria": "inmobiliaria",
            "Lote": "lote",
            "Manzana": "manzana",
            "Fraccionamiento": "fraccionamiento",
            "Direccion": "direccion",
            "Razón social": "razon_social",
            "Comprador": "comprador",
            "Acreedor": "acreedor",
            "Acreedores": "acreedores",
            "Tipo de vivienda": "tipo_vivienda",
            "Categoría": "categoria",
            "Folio real": "folio",
            "Fecha de operacion": "fecha_operacion",
            "Competencia actual": "competencia_actual",
            "Valor de operacion": "valor_operacion",
            "Monto de crédito": "monto_credito",
            "Fecha de partida": "fecha_partida",
            "Partida": "partida",
            "Superficie": "mts_superficie",
            "Edad": "edad",
            "Latitud": "latitud",
            "Longitud": "longitud",
            "Mercado Exe": "mercado_exe",
        },
    )
    .assign(
        folio=lambda df: pd.to_numeric(df["folio"], errors="coerce").astype("Int64"),
        fecha_operacion=lambda df: pd.to_datetime(df["fecha_operacion"]),
        competencia_actual=lambda df: (
            df["competencia_actual"].str.lower().eq("sí").astype("Int64")
        ),
        valor_operacion=lambda df: pd.to_numeric(
            df["valor_operacion"],
            errors="coerce",
        ),
        monto_credito=lambda df: pd.to_numeric(df["monto_credito"], errors="coerce"),
        fecha_partida=lambda df: pd.to_datetime(df["fecha_partida"]),
        partida=lambda df: pd.to_numeric(df["partida"], errors="coerce").astype(
            "Int64",
        ),
        mts_superficie=lambda df: pd.to_numeric(df["mts_superficie"], errors="coerce"),
        edad=lambda df: pd.to_numeric(df["edad"], errors="coerce").astype("Int64"),
        latitud=lambda df: pd.to_numeric(df["latitud"], errors="coerce"),
        longitud=lambda df: pd.to_numeric(df["longitud"], errors="coerce"),
        mercado_exe=lambda df: df["mercado_exe"].str.lower().eq("sí"),
        direccion=lambda df: df["direccion"].astype(str).replace(np.nan, pd.NA),
        fraccionamiento=lambda df: df["fraccionamiento"].astype(str).replace(np.nan, pd.NA),
    )
    .assign(
        fraccionamiento=lambda df: (
            df["fraccionamiento"]
            .str.strip()
            .replace(
                {
                    r"^FRACCIONAMIENTO VILLANOVA.*": "VILLANOVA",
                    r".*PORTICOS DEL VALLE.*": "PORTICOS DEL VALLE",
                },
                regex=True,
            )
            .where(
                lambda x: ~x.str.startswith("FRAC"),
                lambda x: (
                    x.str.replace("FRACCIONAMIENTO", "")
                    .str.replace("FRACC", "")
                    .str.replace("FRACCTO", "")
                    .str.replace("FRACTO", "")
                    .str.strip()
                ),
            )
            .where(
                lambda x: ~x.str.startswith("COLONIA BALBUENA"),
                lambda x: x.str.replace("DE MEXICALI BC DENTRO DE LA", "").str.strip(),
            )
            .where(
                lambda x: ~x.str.startswith("COLONIA GRANJAS AGRICOLAS"),
                lambda x: x.str.replace(
                    "FOLIO REAL 1598810 DE MEXICALI BC",
                    "",
                ).str.strip(),
            )
            .where(
                lambda x: ~x.str.startswith("LA RIOJA"),
                lambda x: x.str.replace("CASTILLAUNA", "CASTILLA").str.strip()
            )
            .str.casefold()
        ),
        coords=lambda df: df["fraccionamiento"].map(
            {
                "angeles de puebla": (32.564770, -115.339935),
                "colonia balbuena": (32.630970, -115.472679),
                "corceles residencial": (32.567668, -115.469794),
                "gran foresta": (32.563446, -115.434414),
                "huertas del colorado": (32.563662, -115.411109),
                "la rioja seccion castillauna": (32.656198, -115.364974),
                "porticos del valle": (32.595718, -115.438633),
                "parajes de puebla": (32.557625, -115.346180),
                "quinta granada": (32.572234, -115.469416),
                "quinta granada 3": (32.567764, -115.473000),
                "san andres": (32.577850, -115.424811),
                "valle oriente": (32.571884, -115.361330),
                "privadas condesa": (32.595546, -115.344834),
            },
        ),
        new_lat=lambda df: df["coords"].apply(
            lambda x: x[0] if not pd.isna(x) else np.nan,
        ),
        new_lon=lambda df: df["coords"].apply(
            lambda x: x[1] if not pd.isna(x) else np.nan,
        ),
        latitud=lambda df: df["latitud"].fillna(df["new_lat"]),
        longitud=lambda df: df["longitud"].fillna(df["new_lon"]),
        privada=lambda df: (
            df["direccion"]
            .str.strip()
            .where(
                lambda x: ~x.str.startswith("DESARROLLO URBANO LA CONDESA"),
                lambda x: x.str.replace(
                    r"DESARROLLO URBANO LA CONDESA SECCI(O|Ó)N",
                    "WANTED_",
                    regex=True,
                ).str.strip(),
            )
            .where(
                lambda x: (
                    ~x.str.startswith(
                        "DESARROLLO URBANO VICTORIA RESIDENCIAL SEGUNDA SECCION",
                    )
                ),
                lambda x: x.str.replace(
                    "DESARROLLO URBANO VICTORIA RESIDENCIAL SEGUNDA SECCION",
                    "WANTED_VICTORIA",
                ).str.strip(),
            )
            .where(lambda x: x.str.startswith("WANTED_"), None)
            .str.replace("WANTED_", "")
            .str.strip()
        ),
        direccion=lambda df: (
            df["direccion"].str.replace("LOCALIZACION:", "").str.strip()
        ),
    )
    .drop(columns=["coords", "new_lat", "new_lon"])
    .query("~fraccionamiento.str.match(r'.*puerto de san felipe.*')")
    .dropna(subset=["fraccionamiento"])
    .reset_index(drop=True)
)

In [37]:
df_init.to_parquet(generated_path / "registro.parquet")